# 06 — Physical scattering and asymptotic safety

Graviton-mediated 2&rarr;2 scalar scattering, RG-improved by the asymptotically-safe
fixed point, and compared against the **physical-scattering bootstrap** of
*Strings from Almost Nothing* (Cheung, Remmen, Sciotti & Tarquini, PRL `cw4p-cqh7`,
[arXiv:2508.09246](https://arxiv.org/abs/2508.09246)).

**Idea.** Promote the running couplings `g(k), lambda(k)` to a momentum-dependent
graviton form factor by identifying the cutoff with the momentum scale,
`k(p^2) = xi*sqrt(|p^2|)` &mdash; the scattering analogue of the RG-improved black
holes in `asymsafety.cosmology`. The amplitude reproduces general relativity in the
infrared and tends to a **finite ultraviolet constant** at the fixed point.

> **Caveat.** This is RG-improvement at the level of an observable, *not* a
> first-principles momentum-dependent form factor. A safe fixed point does not by
> itself guarantee a bounded amplitude (Knorr 2026, [arXiv:2602.21285]).

## 1. Fixed point and RG trajectory

Locate the Reuter non-Gaussian fixed point and integrate a trajectory from the
ultraviolet (near the NGFP) down to the infrared.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from asymsafety.beta.einstein_hilbert import build_eh_beta_system
from asymsafety.analysis.fixed_points import FixedPointFinder
from asymsafety.analysis.flow import FlowIntegrator

system = build_eh_beta_system(d=4)
fp = FixedPointFinder(system).find_fixed_point({'g': 0.7, 'lambda': 0.14})
print('NGFP:', {k: round(v, 4) for k, v in fp.location.items()})

ic_uv = {'g': fp.location['g'] - 1e-3, 'lambda': fp.location['lambda'] + 1e-3}
traj = FlowIntegrator(system).integrate(ic_uv, t_span=(10.0, -10.0), max_step=0.05)
print('trajectory spans t in [%.1f, %.1f]' % (traj.t_values.min(), traj.t_values.max()))

## 2. The graviton form factor

`G(p^2) = g(k(p^2))/k(p^2)^2` softens in the UV; the form factor
`f(p^2) = G_N/G(p^2)` is 1 in the IR and grows like `p^2` in the UV.

In [ ]:
from asymsafety.scattering.form_factor import GravitonFormFactor
from asymsafety.scattering.scale import EnergyScale

ff = GravitonFormFactor(traj, scale=EnergyScale(xi=1.0))
print('Newton constant G_N =', round(ff.newton_constant(), 4))
for p2 in [1e-3, 1.0, 1e3, 1e6]:
    print(f'  p^2={p2:>8.0e}:  f={float(ff.f(p2)):10.3e}   G/G_N={float(ff.G_of_psq(p2))/ff.newton_constant():10.3e}')

## 3. The amplitude: GR in the IR, finite in the UV


In [ ]:
from asymsafety.scattering.amplitude import GravitonMediatedAmplitude
from asymsafety.visualization.amplitude_plot import plot_amplitude_vs_energy

amp = GravitonMediatedAmplitude(ff)
print('IR limit :', amp.ir_limit())
print('UV limit :', amp.uv_limit())

fig = plot_amplitude_vs_energy(amp, s_range=(1e-2, 1e8, 200))
plt.show()

## 4. Physical-scattering consistency

Classical gravity's partial waves grow without bound; the asymptotically-safe
amplitude stays bounded, is UV-finite, ghost-free and crossing symmetric.

In [ ]:
from asymsafety.scattering import consistency as C

report = C.consistency_report(amp)
for key in ('unitarity', 'uv_finiteness', 'froissart', 'causality', 'crossing'):
    print(f'{key:14s} passed={report[key]["passed"]}')
print('all_passed =', report['all_passed'])

## 5. The string bootstrap (Strings from Almost Nothing)

The Veneziano/Virasoro&ndash;Shapiro amplitudes: a Regge tower, crossing symmetry,
and higher-spin residue cancellation.

In [ ]:
from asymsafety.scattering import bootstrap as B

print('open-string spectrum m_n^2 =', B.mass_spectrum(5))
for n in (1, 2, 3):
    print(f'  level {n}: residue zeros at t =', B.residue_zeros(n),
          '(max spin', B.max_spin_at_level(n), ')')

## 6. Bridge: asymptotic safety vs strings

Both satisfy the foundational physical-scattering requirements, but reach UV
completeness differently: asymptotic safety **softens** the coupling (UV-constant),
while strings are **ultrasoft** with an infinite higher-spin tower.

In [ ]:
from asymsafety.scattering.bridge import ScatteringBridge
from asymsafety.visualization.amplitude_plot import plot_as_vs_string

bridge = ScatteringBridge(amp)
verdict = bridge.verify()
print('AS physically consistent :', verdict['as_physically_consistent'])
print('distinct from strings    :', verdict['distinct_from_strings'])
print()
print(verdict['summary'])

fig = plot_as_vs_string(bridge, s_range=(5.0, 60.0, 80))
plt.show()

## Summary

- The asymptotically-safe form factor softens graviton exchange, giving a
  **UV-finite** amplitude that reproduces **Newtonian gravity** in the IR.
- It satisfies the foundational physical-scattering consistency conditions
  (crossing, UV finiteness, no ghosts, bounded partial waves).
- It is a **distinct** consistent point from the string amplitudes singled out
  by the *Strings from Almost Nothing* bootstrap: UV-constant rather than
  ultrasoft, with no infinite Regge tower.